# Experiment Validity

Checks whether the random assignment held and whether the data is internally consistent. Outcomes are never compared across experiment groups or customer segments in this notebook.

In [1]:
import pandas as pd
from scipy.stats import chisquare

df = pd.read_csv("../data/hillstrom.csv")

## Sample ratio mismatch

The challenge description says customers were randomly split into three equal groups. A chi-square goodness-of-fit test compares the observed group sizes with one-third each. A very small p-value would suggest that the assignment or data collection went wrong.

In [2]:
observed = df["segment"].value_counts().sort_index()
expected = pd.Series(len(df) / 3, index=observed.index)
statistic, p_value = chisquare(observed, expected)

display(pd.DataFrame({
    "observed": observed,
    "expected": expected.round(1),
    "difference": (observed - expected).round(1),
    "share": (observed / len(df)).round(4),
}))
print(f"chi-square = {statistic:.3f}, df = {len(observed) - 1}, p = {p_value:.3f}")

,observed,expected,difference,share
segment,,,,
Mens E-Mail,21307,21333.3,-26.3,0.3329
No E-Mail,21306,21333.3,-27.3,0.3329
Womens E-Mail,21387,21333.3,53.7,0.3342


chi-square = 0.203, df = 2, p = 0.904


## Covariate balance

The standardized mean difference (SMD) is the difference between two group means divided by their pooled standard deviation, so it can be compared across features. An absolute value below 0.1 is commonly treated as balanced. Unlike a p-value, it does not shrink simply because the sample is large. Categorical features are split into one 0/1 column per value.

In [3]:
features = pd.concat([
    df[["recency", "history", "mens", "womens", "newbie"]],
    pd.get_dummies(df[["history_segment", "zip_code", "channel"]], prefix_sep=": ", dtype=int),
], axis=1)

means = features.groupby(df["segment"]).mean()
variances = features.groupby(df["segment"]).var()


def smd(group_a, group_b):
    pooled_sd = ((variances.loc[group_a] + variances.loc[group_b]) / 2) ** 0.5
    return (means.loc[group_a] - means.loc[group_b]) / pooled_sd


balance = means.T.round(3)
balance["SMD Mens vs No"] = smd("Mens E-Mail", "No E-Mail").round(4)
balance["SMD Womens vs No"] = smd("Womens E-Mail", "No E-Mail").round(4)
balance["SMD Mens vs Womens"] = smd("Mens E-Mail", "Womens E-Mail").round(4)
smd_columns = balance.columns[-3:]
balance["Largest |SMD|"] = balance[smd_columns].abs().max(axis=1)

print(f"features checked: {len(balance)}, comparisons: {len(balance) * len(smd_columns)}")
print(f"largest |SMD|: {balance['Largest |SMD|'].max():.4f}")
print(f"features at or above 0.1: {(balance['Largest |SMD|'] >= 0.1).sum()}")
balance

features checked: 18, comparisons: 54
largest |SMD|: 0.0169
features at or above 0.1: 0


segment,Mens E-Mail,No E-Mail,Womens E-Mail,SMD Mens vs No,SMD Womens vs No,SMD Mens vs Womens,Largest |SMD|
recency,5.774,5.750,5.768,0.0068,0.0052,0.0016,0.0068
history,242.836,240.883,242.537,0.0076,0.0065,0.0012,0.0076
mens,0.551,0.553,0.549,-0.0046,-0.0086,0.0040,0.0086
womens,0.551,0.548,0.550,0.0076,0.0049,0.0026,0.0076
newbie,0.502,0.502,0.503,-0.0009,0.0026,-0.0034,0.0034
history_segment: 1) $0 - $100,0.363,0.357,0.357,0.0109,-0.0007,0.0116,0.0116
history_segment: 2) $100 - $200,0.220,0.227,0.221,-0.0164,-0.0143,-0.0021,0.0164
history_segment: 3) $200 - $350,0.192,0.190,0.194,0.0055,0.0114,-0.0059,0.0114
history_segment: 4) $350 - $500,0.098,0.100,0.102,-0.0043,0.0087,-0.0129,0.0129
history_segment: 5) $500 - $750,0.075,0.078,0.078,-0.0097,0.0006,-0.0104,0.0104


## Data checks

Each row counts the records that fail or match a check. Outcome columns are used only to test their internal logic.

In [4]:
binary = ["mens", "womens", "newbie", "visit", "conversion"]
segment_bounds = df["history_segment"].str.replace(",", "").str.extract(r"\$(\d+)(?: - \$(\d+))?").astype(float)
segment_low, segment_high = segment_bounds[0], segment_bounds[1].fillna(float("inf"))

checks = {
    "Missing values": df.isna().sum().sum(),
    "Exact repeat rows (reviewed and kept)": df.duplicated().sum(),
    "conversion = 1 but visit = 0": ((df["conversion"] == 1) & (df["visit"] == 0)).sum(),
    "spend > 0 but conversion = 0": ((df["spend"] > 0) & (df["conversion"] == 0)).sum(),
    "conversion = 1 but spend = 0": ((df["conversion"] == 1) & (df["spend"] == 0)).sum(),
    "spend < 0": (df["spend"] < 0).sum(),
    "recency outside 1 to 12": (~df["recency"].between(1, 12)).sum(),
    "Binary column not 0 or 1": (~df[binary].isin([0, 1])).sum().sum(),
    "mens = 0 and womens = 0": ((df["mens"] == 0) & (df["womens"] == 0)).sum(),
    "history outside its history_segment range": ((df["history"] < segment_low) | (df["history"] > segment_high)).sum(),
    "Unexpected experiment group": (~df["segment"].isin(observed.index)).sum(),
}
pd.Series(checks, name="rows").astype(int).to_frame()

,rows
Missing values,0
Exact repeat rows (reviewed and kept),6562
conversion = 1 but visit = 0,0
spend > 0 but conversion = 0,0
conversion = 1 but spend = 0,0
spend < 0,0
recency outside 1 to 12,0
Binary column not 0 or 1,0
mens = 0 and womens = 0,0
history outside its history_segment range,0
